In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Task 1 — Extract
# MAGIC Fetches the source page and saves it, untouched, into the bronze volume as `page_<run_date>.html`.
# MAGIC
# MAGIC Nothing is parsed here. Keeping the raw page means the transform task can be re-run or fixed
# MAGIC later without scraping the site again.
# MAGIC
# MAGIC **Re-running the same `run_date` overwrites that date's file**, so the task can be repeated safely.

# COMMAND ----------

# ============================================================
# 1. PARAMETERS
# ============================================================

dbutils.widgets.text("catalog", "gdp_etl_catalog")
dbutils.widgets.text("run_date", "")          # set by the job; blank = today (UTC)
dbutils.widgets.text("source_url", "https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(nominal)")

CATALOG    = dbutils.widgets.get("catalog")
SOURCE_URL = dbutils.widgets.get("source_url")

BRONZE_DIR = f"/Volumes/{CATALOG}/bronze/raw_html"
REQUEST_HEADERS = {"User-Agent": "GDP-ETL-Portfolio/1.0"}

# COMMAND ----------

# ============================================================
# 2. IMPORTS
# ============================================================

import os
from datetime import date, datetime, timezone

import requests

RUN_DATE = date.fromisoformat(dbutils.widgets.get("run_date")) if dbutils.widgets.get("run_date") \
    else datetime.now(timezone.utc).date()

STAGE = "extract"

# COMMAND ----------

# ============================================================
# 3. FUNCTIONS
# ============================================================

LOG_LINES = []


def write_file(path, content):
    """Write a text file to a volume, replacing anything already at that path.

    Volumes don't allow appending to or editing a file in place, so a re-run has to
    delete first. Without this, the second run of a date fails with 'Illegal seek'.
    """
    if os.path.exists(path):
        os.remove(path)
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)


def log_progress(message):
    """Collect a timestamped message and print it. Returns nothing."""
    line = f"{datetime.now(timezone.utc):%Y-%m-%d %H:%M:%S} : [{STAGE}] {message}"
    LOG_LINES.append(line)
    print(line)


def write_log(log_dir, run_date, stage):
    """Save this task's collected log lines as one file."""
    path = f"{log_dir}/etl_log_{run_date}_{stage}.txt"
    write_file(path, "\n".join(LOG_LINES) + "\n")
    print(f"Log written to {path}")

# COMMAND ----------

# ============================================================
# 4. TASK EXECUTION
# ============================================================

log_progress(f"Initiating extraction (run_date {RUN_DATE})")

response = requests.get(SOURCE_URL, headers=REQUEST_HEADERS, timeout=30)
response.raise_for_status()
log_progress(f"Page fetched from {SOURCE_URL} ({len(response.text)} characters)")

raw_path = f"{BRONZE_DIR}/page_{RUN_DATE}.html"
write_file(raw_path, response.text)
log_progress(f"Raw page saved to {raw_path}")

log_progress("Extraction complete")
write_log(BRONZE_DIR, RUN_DATE, STAGE)